In [ ]:
# Malariagen API arguments for each call below.
# Replace a value to override it; keep the marker to use the call-site
# value or the method's default.
API_KEEP_CALLSITE_VALUE = '__KEEP_CALLSITE_VALUE__'
PERFORMANCE_CSV_PATH = 'auto_chunks_performance.csv'
API_CALL_PARAMETERS = {'cell_4_call_1_ag3_snp_calls': {'region': '3R',
                                 'sample_sets': 'AG1000G-BF-A',
                                 'sample_query': None,
                                 'sample_query_options': None,
                                 'sample_indices': None,
                                 'site_mask': None,
                                 'site_class': None,
                                 'inline_array': True,
                                 'chunks': 'native',
                                 'cohort_size': None,
                                 'min_cohort_size': None,
                                 'max_cohort_size': None,
                                 'random_seed': 42},
 'cell_8_call_1_ag3_snp_calls': {'region': '3R',
                                 'sample_sets': 'AG1000G-BF-A',
                                 'sample_query': None,
                                 'sample_query_options': None,
                                 'sample_indices': None,
                                 'site_mask': None,
                                 'site_class': None,
                                 'inline_array': True,
                                 'chunks': '300MB',
                                 'cohort_size': None,
                                 'min_cohort_size': None,
                                 'max_cohort_size': None,
                                 'random_seed': 42},
 'cell_12_call_1_ag3_snp_calls': {'region': '3R',
                                  'sample_sets': 'AG1000G-BF-A',
                                  'sample_query': None,
                                  'sample_query_options': None,
                                  'sample_indices': None,
                                  'site_mask': None,
                                  'site_class': None,
                                  'inline_array': True,
                                  'chunks': 'auto',
                                  'cohort_size': None,
                                  'min_cohort_size': None,
                                  'max_cohort_size': None,
                                  'random_seed': 42},
 'cell_16_call_1_ag3_snp_calls': {'region': '3R',
                                  'sample_sets': 'AG1000G-BF-A',
                                  'sample_query': None,
                                  'sample_query_options': None,
                                  'sample_indices': None,
                                  'site_mask': None,
                                  'site_class': None,
                                  'inline_array': True,
                                  'chunks': 'ndauto',
                                  'cohort_size': None,
                                  'min_cohort_size': None,
                                  'max_cohort_size': None,
                                  'random_seed': 42}}

In [ ]:
import inspect
from memory_profiler import memory_usage
import time
import json
import csv
from pathlib import Path

def _parameter_value_for_csv(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): _parameter_value_for_csv(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_parameter_value_for_csv(item) for item in value]
    shape = getattr(value, "shape", None)
    if shape is not None:
        return {
            "type": f"{type(value).__module__}.{type(value).__qualname__}",
            "shape": list(shape),
        }
    return repr(value)


import collections.abc
import types
import typing


def _tuple_if_expected(annotation, value):
    """Turn a list back into a tuple where the API accepts a tuple but not a
    list, e.g. percentiles. CSV configs store values as JSON, which has no
    tuples, so (50, 75, 100) arrives as [50, 75, 100]."""
    if not isinstance(value, list):
        return value
    if typing.get_origin(annotation) is typing.Annotated:
        annotation = typing.get_args(annotation)[0]
    if typing.get_origin(annotation) in (typing.Union, types.UnionType):
        options = typing.get_args(annotation)
    else:
        options = (annotation,)
    accepted = {typing.get_origin(option) or option for option in options}
    accepts_list = accepted & {
        list, typing.Any, collections.abc.Sequence, collections.abc.Iterable,
        collections.abc.Collection, collections.abc.MutableSequence,
    }
    if tuple in accepted and not accepts_list:
        return tuple(value)
    return value


def _call_api_with_parameters(method, call_key, *args, **kwargs):
    signature = inspect.signature(method)
    bound = signature.bind_partial(*args, **kwargs)
    bound.apply_defaults()
    for name, value in API_CALL_PARAMETERS[call_key].items():
        if not (isinstance(value, str) and value == API_KEEP_CALLSITE_VALUE):
            parameter = signature.parameters.get(name)
            if parameter is not None:
                value = _tuple_if_expected(parameter.annotation, value)
            bound.arguments[name] = value

    positional = []
    keyword = {}
    for parameter in signature.parameters.values():
        if parameter.kind is inspect.Parameter.POSITIONAL_ONLY:
            if parameter.name in bound.arguments:
                positional.append(bound.arguments[parameter.name])
        elif parameter.kind is inspect.Parameter.POSITIONAL_OR_KEYWORD:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_POSITIONAL:
            positional.extend(bound.arguments.get(parameter.name, ()))
        elif parameter.kind is inspect.Parameter.KEYWORD_ONLY:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_KEYWORD:
            keyword.update(bound.arguments.get(parameter.name, {}))
    result = {}

    def operation():
        result["value"] = method(*positional, **keyword)

    start_time = time.perf_counter()
    peak_memory_mib = memory_usage((operation, ()), max_usage=True)
    runtime_sec = time.perf_counter() - start_time

    output_path = Path(PERFORMANCE_CSV_PATH)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "call_key",
        "method",
        "parameters_json",
        "runtime_sec",
        "peak_memory_mib",
    ]
    record = {
        "call_key": call_key,
        "method": getattr(method, "__qualname__", getattr(method, "__name__", repr(method))),
        "parameters_json": json.dumps(
            {
                name: _parameter_value_for_csv(value)
                for name, value in bound.arguments.items()
            },
            sort_keys=True,
        ),
        "runtime_sec": round(runtime_sec, 4),
        "peak_memory_mib": round(peak_memory_mib, 2),
    }
    file_exists = output_path.exists() and output_path.stat().st_size > 0
    with output_path.open("a", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(record)

    return result["value"]

In [ ]:
import malariagen_data

In [ ]:
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../gcs_cache"),
)
ag3

## Default chunk sizes

In [ ]:
ds = _call_api_with_parameters(ag3.snp_calls, 'cell_4_call_1_ag3_snp_calls', region="3R", sample_sets="AG1000G-BF-A")

In [ ]:
ds["variant_position"].data

In [ ]:
ds["call_genotype"].data

## Explicit chunk sizes

In [ ]:
ds = _call_api_with_parameters(ag3.snp_calls, 'cell_8_call_1_ag3_snp_calls', region="3R", sample_sets="AG1000G-BF-A", chunks="300MB")

In [ ]:
ds["variant_position"].data

In [ ]:
ds["call_genotype"].data

## Auto chunk sizes

In [ ]:
ds = _call_api_with_parameters(ag3.snp_calls, 'cell_12_call_1_ag3_snp_calls', region="3R", sample_sets="AG1000G-BF-A", chunks="auto")

In [ ]:
ds["variant_position"].data

In [ ]:
ds["call_genotype"].data

## ND auto chunk sizes

In [ ]:
ds = _call_api_with_parameters(ag3.snp_calls, 'cell_16_call_1_ag3_snp_calls', region="3R", sample_sets="AG1000G-BF-A", chunks="ndauto")

In [ ]:
ds["variant_position"].data

In [ ]:
ds["call_genotype"].data